# Step 05 — Traffic Quality Gate & Corridor Probe Planning


In [17]:
import copy
import json
import pickle
import shutil
from datetime import datetime, timezone

import networkx as nx
import pandas as pd
from IPython.display import display

# Import stable project settings separately from optional probe controls.
from routing.config import (
    RESULTS_DIR, ORIGIN, ORIGINS, DESTINATION, MAP_RADIUS_M,
    MAX_TRAFFIC_REQUESTS, MIN_TOMTOM_CONFIDENCE, TRAFFIC_REQUEST_PAUSE_S,
)
import routing.config as routing_config

# These defaults keep Notebook 05 compatible with an older config.py.
# If the values exist in src/routing/config.py, those values take precedence.
MIN_PROBE_EDGE_LENGTH_M = float(
    getattr(routing_config, "MIN_PROBE_EDGE_LENGTH_M", 35.0)
)
MAX_PROBE_ATTEMPTS_PER_EDGE = int(
    getattr(routing_config, "MAX_PROBE_ATTEMPTS_PER_EDGE", 3)
)
PROBE_FRACTIONS = tuple(
    getattr(routing_config, "PROBE_FRACTIONS", (0.50, 0.25, 0.75))
)

if MIN_PROBE_EDGE_LENGTH_M < 0:
    raise ValueError("MIN_PROBE_EDGE_LENGTH_M must be non-negative")
if MAX_PROBE_ATTEMPTS_PER_EDGE < 1:
    raise ValueError("MAX_PROBE_ATTEMPTS_PER_EDGE must be at least 1")
if not PROBE_FRACTIONS or any(not 0.0 < float(x) < 1.0 for x in PROBE_FRACTIONS):
    raise ValueError("PROBE_FRACTIONS must contain values strictly between 0 and 1")

from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import (
    collect_traffic_snapshot,
    make_routing_graph,
    preview_probes,
    save_summary,
)
from routing.traffic_quality import (
    plan_corridor_probes,
    audit_traffic_snapshot,
)
from routing.probe_history import (
    load_probe_history,
    update_probe_history,
    save_probe_history,
)

RADIUS_M = MAP_RADIUS_M
G = load_road_graph(radius_m=RADIUS_M)

assert isinstance(G, nx.MultiDiGraph)
assert G.is_directed()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("========== NOTEBOOK 05 INITIALIZED ==========")
print("Graph:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Directed edges:", G.number_of_edges())
print("Results directory:", RESULTS_DIR)
print("Minimum probe edge length (m):", MIN_PROBE_EDGE_LENGTH_M)
print("Maximum attempts per edge:", MAX_PROBE_ATTEMPTS_PER_EDGE)
print("Probe fractions:", PROBE_FRACTIONS)
print("PASS: Imports, probe settings, and OSM graph loaded")


========== NOTEBOOK 05 INITIALIZED ==========
Graph: MultiDiGraph
Nodes: 2569
Directed edges: 5850
Results directory: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results
Minimum probe edge length (m): 35.0
Maximum attempts per edge: 3
Probe fractions: (0.5, 0.25, 0.75)
PASS: Imports, probe settings, and OSM graph loaded


## 1 — Load and protect the existing Snapshot


In [18]:

snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"

previous_bundle = None
previous_snapshot = None
previous_summary = {}

def _assert_same_topology(base_graph, candidate_graph, label):
    if not isinstance(candidate_graph, nx.MultiDiGraph):
        raise TypeError(f"{label} must be a networkx.MultiDiGraph")

    if set(base_graph.nodes) != set(candidate_graph.nodes):
        raise ValueError(f"{label} node set differs from the current OSM graph")

    if set(base_graph.edges(keys=True)) != set(candidate_graph.edges(keys=True)):
        raise ValueError(f"{label} directed-edge set differs from the current OSM graph")


if snapshot_path.exists():
    backup_path = RESULTS_DIR / (
        "traffic_snapshot_before_merge_"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        + ".pkl"
    )
    shutil.copy2(snapshot_path, backup_path)

    with snapshot_path.open("rb") as f:
        raw_bundle = pickle.load(f)

    # Support both the current bundle format and an older raw-graph file.
    if isinstance(raw_bundle, dict):
        if "snapshot" not in raw_bundle:
            raise ValueError(
                "traffic_snapshot.pkl is a dictionary but has no 'snapshot' key."
            )
        previous_bundle = raw_bundle
        previous_snapshot = raw_bundle["snapshot"]

    elif isinstance(raw_bundle, nx.MultiDiGraph):
        previous_snapshot = raw_bundle
        previous_bundle = {"snapshot": raw_bundle, "summary": {}}

    else:
        raise TypeError(
            "Unsupported traffic_snapshot.pkl format: "
            f"{type(raw_bundle).__name__}"
        )

    _assert_same_topology(G, previous_snapshot, "saved snapshot")
    previous_summary = previous_bundle.get("summary", {})

    print("Existing snapshot loaded:", snapshot_path)
    print("Backup created:", backup_path)
    print("Nodes:", previous_snapshot.number_of_nodes())
    print("Edges:", previous_snapshot.number_of_edges())

else:
    print("No previous snapshot exists. A new one will be created after live collection.")

print("PASS: Snapshot loading/topology validation completed")


Existing snapshot loaded: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl
Backup created: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot_before_merge_20261109_105318.pkl
Nodes: 2569
Edges: 5850
PASS: Snapshot loading/topology validation completed


## 2 — Plan corridor probes


In [19]:

history = load_probe_history()

REQUEST_LIMIT = min(12, MAX_TRAFFIC_REQUESTS)

plan = plan_corridor_probes(
    G,
    max_probes=REQUEST_LIMIT,
    origins=ORIGINS,
    destination=DESTINATION,
    corridor_width_m=350,
    min_length_m=35,
    min_spacing_m=65,
    history=history,
)

plan_df = pd.DataFrame(plan)

print("Maximum corridor probes:", REQUEST_LIMIT)
print("Planned corridor probes:", len(plan))

if not plan_df.empty and "road_class" in plan_df.columns:
    print("\nRoad-class distribution:")
    display(plan_df["road_class"].value_counts().rename("count").to_frame())

display(plan_df)


Maximum corridor probes: 12
Planned corridor probes: 12

Road-class distribution:


,count
road_class,
trunk,8
primary,4


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(26389013, 1116949690, 0)",51.495245,-0.180497,65.81,trunk,Gloucester Road,85.8
1,"(20947178, 1691189798, 0)",51.501582,-0.177494,89.35,primary,Royal Albert Hall,77.2
2,"(2180741853, 2360746811, 0)",51.495563,-0.176366,180.03,trunk,South Kensington,150.4
3,"(5898890847, 4749990829, 0)",51.494899,-0.184022,115.84,trunk,Gloucester Road,114.7
4,"(385312258, 20947178, 0)",51.501568,-0.178813,93.45,primary,Royal Albert Hall,107.2
5,"(3818359076, 3818359130, 0)",51.496322,-0.170391,174.38,trunk,South Kensington,264.5
6,"(1116949690, 2180739560, 0)",51.495316,-0.179549,66.41,trunk,Gloucester Road,117.6
7,"(1691189798, 109670, 0)",51.501654,-0.175713,158.14,primary,Royal Albert Hall,153.8
8,"(2180741854, 2180744124, 0)",51.495331,-0.178271,88.53,trunk,South Kensington,282.7
9,"(1260002921, 5940891836, 0)",51.494879,-0.185425,73.51,trunk,Gloucester Road,208.4


## 3 — Select route-focused probes



In [20]:
USE_ROUTE_FOCUSED_PROBES = True  # Route candidates are prioritized, not exclusive.
REQUEST_LIMIT = min(12, MAX_TRAFFIC_REQUESTS)
BATCH_SIZE = REQUEST_LIMIT

candidate_records = []
pending_probe_records = []
selected_probe_records = []
selected_edge_ids = []
selected_probe_fractions = {}

route_probe_path = RESULTS_DIR / "route_probe_candidates.json"
route_candidate_records = []
stale_route_candidates = 0

# A third source spreads probes across the full downloaded map, not only the
# selected A* route or the configured origin/destination corridors.
area_candidate_records = preview_probes(G, limit=MAX_TRAFFIC_REQUESTS)

if USE_ROUTE_FOCUSED_PROBES and route_probe_path.exists():
    try:
        loaded_candidates = json.loads(route_probe_path.read_text(encoding="utf-8"))
        if isinstance(loaded_candidates, list):
            route_candidate_records = loaded_candidates
        else:
            print("Route candidate file is not a list; it will be ignored.")
    except (OSError, json.JSONDecodeError) as exc:
        print("Could not read route candidate file; corridor candidates will still be used:", exc)

# Keep A* route edges first, then append the broader corridor plan.
# Duplicate edge IDs are removed, and stale route IDs are ignored safely.
candidate_sources = []
if USE_ROUTE_FOCUSED_PROBES:
    candidate_sources.append(("A* route", route_candidate_records))
candidate_sources.append(("corridor", plan))
candidate_sources.append(("area coverage", area_candidate_records))

seen_candidate_ids = set()
for source_name, records in candidate_sources:
    for record in records:
        try:
            raw_edge_id = record["edge_id"]
            edge_id = tuple(int(value) for value in raw_edge_id)
            if len(edge_id) != 3:
                continue
        except (KeyError, TypeError, ValueError, OverflowError):
            continue

        if not G.has_edge(*edge_id):
            stale_route_candidates += source_name == "A* route"
            continue
        if edge_id in seen_candidate_ids:
            continue

        seen_candidate_ids.add(edge_id)
        normalized = dict(record)
        normalized["edge_id"] = edge_id
        normalized["probe_source"] = source_name
        candidate_records.append(normalized)

usable_statuses = {"red", "yellow", "clear", "closed"}

def _has_usable_observation(attrs):
    return (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status") in usable_statuses
    )

def _edge_history_attempts(edge_id):
    edge_history = history.get("edges", {}) if isinstance(history, dict) else {}
    item = edge_history.get("|".join(map(str, edge_id)), {})
    try:
        attempts = max(0, int(item.get("attempts", 0)))
        global_errors = max(0, int(item.get("global_errors", 0)))
        return max(0, attempts - global_errors)
    except (AttributeError, TypeError, ValueError, OverflowError):
        return 0

pending_probe_records = []
seen_edge_ids = set()
skipped_observed = 0
skipped_short = 0
skipped_max_attempts = 0

for record in candidate_records:
    edge_id = tuple(record["edge_id"])
    if edge_id in seen_edge_ids:
        continue
    seen_edge_ids.add(edge_id)

    u, v, key = edge_id
    osm_attrs = G[u][v][key]
    try:
        edge_length = float(record.get("length_m", osm_attrs.get("length", 0.0)))
    except (TypeError, ValueError, OverflowError):
        edge_length = 0.0

    if edge_length < MIN_PROBE_EDGE_LENGTH_M:
        skipped_short += 1
        continue

    attrs = (
        previous_snapshot[u][v][key]
        if previous_snapshot is not None
        else osm_attrs
    )

    if _has_usable_observation(attrs):
        skipped_observed += 1
        continue

    history_item = history.get("edges", {}).get("|".join(map(str, edge_id)), {})
    try:
        stored_attempts = int(attrs.get("traffic_attempt_count", 0) or 0)
        global_errors = int(history_item.get("global_errors", 0) or 0)
        stored_attempts = max(0, stored_attempts - max(0, global_errors))
    except (AttributeError, TypeError, ValueError, OverflowError):
        stored_attempts = 0

    attempts = max(stored_attempts, _edge_history_attempts(edge_id))
    if attempts >= MAX_PROBE_ATTEMPTS_PER_EDGE:
        skipped_max_attempts += 1
        continue

    fraction_index = min(attempts, len(PROBE_FRACTIONS) - 1)
    probe_fraction = float(PROBE_FRACTIONS[fraction_index])

    pending_probe_records.append({
        **record,
        "edge_id": edge_id,
        "length_m": round(edge_length, 2),
        "probe_fraction": probe_fraction,
        "attempt_number": attempts + 1,
        "previous_failure": attrs.get("traffic_last_attempt_reason")
            or attrs.get("traffic_reason"),
    })

selected_probe_records = pending_probe_records[:BATCH_SIZE]
selected_edge_ids = [tuple(record["edge_id"]) for record in selected_probe_records]
selected_probe_fractions = {
    tuple(record["edge_id"]): float(record["probe_fraction"])
    for record in selected_probe_records
}

print("A* route candidates:", len(route_candidate_records))
print("Corridor candidates:", len(plan))
print("Spatially distributed area candidates:", len(area_candidate_records))
print("Combined unique candidates:", len(candidate_records))
print("Stale route candidates ignored:", stale_route_candidates)
print("Already observed and skipped:", skipped_observed)
print("Too-short edges skipped:", skipped_short)
print("Maximum-attempt edges skipped:", skipped_max_attempts)
print("Pending candidates:", len(pending_probe_records))
print("Selected for this run:", len(selected_edge_ids))
print("Maximum requests this run:", BATCH_SIZE)

if selected_probe_records:
    display(pd.DataFrame(selected_probe_records))
else:
    print("No eligible probes remain. The previous Snapshot will be retained; no API call is needed.")

assert len(selected_edge_ids) <= BATCH_SIZE
assert len(selected_edge_ids) == len(set(selected_edge_ids))

A* route candidates: 3
Corridor candidates: 12
Spatially distributed area candidates: 60
Combined unique candidates: 74
Stale route candidates ignored: 0
Already observed and skipped: 6
Too-short edges skipped: 0
Maximum-attempt edges skipped: 3
Pending candidates: 65
Selected for this run: 12
Maximum requests this run: 12


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m,probe_source,probe_fraction,attempt_number,previous_failure
0,"(26389013, 1116949690, 0)",51.495245,-0.180497,65.81,trunk,Gloucester Road,85.8,corridor,0.5,1,not_queried
1,"(2180741853, 2360746811, 0)",51.495563,-0.176366,180.03,trunk,South Kensington,150.4,corridor,0.5,1,not_queried
2,"(5898890847, 4749990829, 0)",51.494899,-0.184022,115.84,trunk,Gloucester Road,114.7,corridor,0.5,1,not_queried
3,"(3818359076, 3818359130, 0)",51.496322,-0.170391,174.38,trunk,South Kensington,264.5,corridor,0.5,1,not_queried
4,"(1116949690, 2180739560, 0)",51.495316,-0.179549,66.41,trunk,Gloucester Road,117.6,corridor,0.5,1,not_queried
5,"(2180741854, 2180744124, 0)",51.495331,-0.178271,88.53,trunk,South Kensington,282.7,corridor,0.5,1,not_queried
6,"(1260002921, 5940891836, 0)",51.494879,-0.185425,73.51,trunk,Gloucester Road,208.4,corridor,0.5,1,not_queried
7,"(276548, 3764277424, 0)",51.500031,-0.174850,56.30,NaN,NaN,NaN,area coverage,0.5,1,not_queried
8,"(776736207, 776736265, 0)",51.482205,-0.204049,39.10,NaN,NaN,NaN,area coverage,0.5,1,not_queried
9,"(154176677, 255468062, 0)",51.517113,-0.204258,72.80,NaN,NaN,NaN,area coverage,0.5,1,not_queried


## 4 — Merge helper: preserve older valid observations



In [21]:
_TRAFFIC_VALUE_FIELDS = {
    "current_speed_kph",
    "free_flow_speed_kph",
    "travel_time_s",
    "free_flow_time_s",
    "time_is_imputed",
    "free_flow_is_imputed",
    "fallback_speed_kph",
    "is_closed",
}

def _is_traffic_attribute(name):
    return name.startswith("traffic_") or name in _TRAFFIC_VALUE_FIELDS

def _is_usable_observation(attrs):
    return (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status") in {"red", "yellow", "clear", "closed"}
    )

def merge_traffic_snapshots(base_graph, previous_snapshot, fresh_snapshot, checked_at_utc):
    """Merge one fresh batch while preserving older valid observations."""
    _assert_same_topology(base_graph, fresh_snapshot, "fresh_snapshot")

    if previous_snapshot is None:
        merged = base_graph.copy()
    else:
        _assert_same_topology(base_graph, previous_snapshot, "previous_snapshot")
        merged = previous_snapshot.copy()

    refreshed_edges = 0
    for u, v, key, fresh_attrs in fresh_snapshot.edges(keys=True, data=True):
        was_assessed = (
            bool(fresh_attrs.get("traffic_directly_queried"))
            or bool(fresh_attrs.get("traffic_sampled"))
            or fresh_attrs.get("traffic_match_source") == "reverse_inferred"
        )
        if not was_assessed:
            continue

        target_attrs = merged[u][v][key]
        old_is_usable = _is_usable_observation(target_attrs)
        fresh_is_usable = _is_usable_observation(fresh_attrs)

        old_attempt_count = target_attrs.get("traffic_attempt_count")
        if old_attempt_count is None:
            old_attempt_count = 1 if target_attrs.get("traffic_directly_queried") is True else 0
        try:
            old_attempt_count = max(0, int(old_attempt_count))
        except (TypeError, ValueError, OverflowError):
            old_attempt_count = 0

        if fresh_is_usable or not old_is_usable:
            for attr_name in list(target_attrs):
                if _is_traffic_attribute(attr_name):
                    target_attrs.pop(attr_name, None)
            for attr_name, value in fresh_attrs.items():
                if _is_traffic_attribute(attr_name):
                    target_attrs[attr_name] = copy.deepcopy(value)
        else:
            # Keep the previous speed/time, but preserve all diagnostics from
            # the latest failed attempt separately.
            target_attrs["traffic_last_attempt_reason"] = fresh_attrs.get("traffic_reason")
            target_attrs["traffic_last_attempt_match_reason"] = fresh_attrs.get("traffic_match_reason")
            target_attrs["traffic_last_attempt_midpoint_distance_m"] = fresh_attrs.get("traffic_match_midpoint_distance_m")
            target_attrs["traffic_last_attempt_fraction_near"] = fresh_attrs.get("traffic_match_fraction_near")
            target_attrs["traffic_last_attempt_direction_cosine"] = fresh_attrs.get("traffic_match_direction_cosine")
            target_attrs["traffic_last_probe_fraction"] = fresh_attrs.get("traffic_probe_fraction")
            target_attrs["traffic_last_probe_latitude"] = fresh_attrs.get("traffic_probe_latitude")
            target_attrs["traffic_last_probe_longitude"] = fresh_attrs.get("traffic_probe_longitude")
            target_attrs["traffic_last_attempt_matched"] = bool(fresh_attrs.get("traffic_matched"))

        if fresh_attrs.get("traffic_directly_queried") is True:
            target_attrs["traffic_attempt_count"] = old_attempt_count + 1
        else:
            target_attrs.setdefault("traffic_attempt_count", old_attempt_count)

        target_attrs["traffic_last_checked_at_utc"] = checked_at_utc
        if fresh_is_usable:
            target_attrs["traffic_observed_at_utc"] = checked_at_utc

        refreshed_edges += 1

    merged.graph.update(copy.deepcopy(fresh_snapshot.graph))
    merged.graph["traffic_last_merge_refreshed_edges"] = refreshed_edges
    return merged, refreshed_edges


def count_usable_observations(graph):
    usable = 0
    sampled = 0
    direct = 0
    reverse = 0

    for _, _, _, attrs in graph.edges(keys=True, data=True):
        sampled += bool(attrs.get("traffic_sampled"))
        if _is_usable_observation(attrs):
            usable += 1
            if attrs.get("traffic_match_source") == "direct_probe":
                direct += 1
            elif attrs.get("traffic_match_source") == "reverse_inferred":
                reverse += 1

    total = graph.number_of_edges()
    coverage = 100.0 * usable / total if total else 0.0
    return {
        "usable_observed_edges": usable,
        "direct_observed_edges": direct,
        "reverse_inferred_observed_edges": reverse,
        "sampled_directed_edges": sampled,
        "usable_coverage_percent": round(coverage, 3),
    }

print("PASS: Snapshot merge helpers are ready")


PASS: Snapshot merge helpers are ready


## 5 — Collect live traffic or load the previous snapshot



In [22]:
RUN_LIVE_TRAFFIC = True  # Change to True only when you intentionally want live TomTom requests.

fresh_snapshot = None
batch_summary = None
refreshed_edges = 0

if "selected_edge_ids" not in globals() or "selected_probe_records" not in globals():
    raise RuntimeError(
        "Probe-selection variables are missing. Run Notebook 05 cells from the top."
    )

if RUN_LIVE_TRAFFIC and selected_edge_ids:
    with TomTomClient() as client:
        fresh_snapshot, batch_summary = collect_traffic_snapshot(
            osm_graph=G,
            client=client,
            max_requests=len(selected_edge_ids),
            selected_edge_ids=selected_edge_ids,
            probe_fractions=selected_probe_fractions,
            pause_s=TRAFFIC_REQUEST_PAUSE_S,
            confidence_threshold=MIN_TOMTOM_CONFIDENCE,
        )

    checked_at = batch_summary.get("finished_at_utc") or datetime.now(timezone.utc).isoformat()
    snapshot, refreshed_edges = merge_traffic_snapshots(
        G,
        previous_snapshot,
        fresh_snapshot,
        checked_at,
    )

    summary = dict(batch_summary)
    cumulative = count_usable_observations(snapshot)
    summary["merged_refreshed_edges"] = refreshed_edges
    summary["cumulative_usable_observed_edges"] = cumulative["usable_observed_edges"]
    summary["cumulative_direct_observed_edges"] = cumulative["direct_observed_edges"]
    summary["cumulative_reverse_inferred_observed_edges"] = cumulative["reverse_inferred_observed_edges"]
    summary["cumulative_sampled_directed_edges"] = cumulative["sampled_directed_edges"]
    summary["cumulative_usable_coverage_percent"] = cumulative["usable_coverage_percent"]

    print("========== LIVE COLLECTION COMPLETED ==========")
    print("Requests made:", summary.get("requests_made"))
    print("Request errors:", summary.get("request_errors"))
    print("Newly refreshed directed edges:", refreshed_edges)
    print("Cumulative usable observations:", cumulative["usable_observed_edges"])

else:
    if previous_snapshot is None:
        snapshot, summary = collect_traffic_snapshot(
            G, client=None, max_requests=0, pause_s=0
        )
        print("No saved snapshot exists. Created a temporary offline snapshot; it will not be saved as live traffic.")
    else:
        snapshot = previous_snapshot
        summary = dict(previous_summary)
        print("Loaded existing snapshot; no API request was made.")

    if RUN_LIVE_TRAFFIC and not selected_edge_ids:
        print("No eligible probes remain. No TomTom request was sent.")

_assert_same_topology(G, snapshot, "working snapshot")
print("Snapshot nodes:", snapshot.number_of_nodes())
print("Snapshot directed edges:", snapshot.number_of_edges())
print("Cumulative observation counts:", count_usable_observations(snapshot))

========== LIVE COLLECTION COMPLETED ==========
Requests made: 12
Request errors: 0
Newly refreshed directed edges: 14
Cumulative usable observations: 33
Snapshot nodes: 2569
Snapshot directed edges: 5850
Cumulative observation counts: {'usable_observed_edges': 33, 'direct_observed_edges': 24, 'reverse_inferred_observed_edges': 9, 'sampled_directed_edges': 57, 'usable_coverage_percent': 0.564}


## 6 — Inspect the selected probes against the merged Snapshot

In [23]:

probe_rows = []

for item in selected_probe_records:
    edge_id = tuple(item["edge_id"])
    u, v, key = edge_id
    attrs = snapshot[u][v][key]

    probe_rows.append({
        "edge_id": edge_id,
        "road_class": item.get("road_class", attrs.get("highway")),
        "sampled": attrs.get("traffic_sampled", False),
        "directly_queried": attrs.get("traffic_directly_queried", False),
        "matched": attrs.get("traffic_matched", False),
        "match_source": attrs.get("traffic_match_source"),
        "match_reason": attrs.get("traffic_match_reason"),
        "traffic_status": attrs.get("traffic_status", "missing"),
        "traffic_reason": attrs.get("traffic_reason"),
        "current_speed_kph": attrs.get("current_speed_kph"),
        "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
        "confidence": attrs.get("traffic_confidence"),
        "match_distance_m": attrs.get("traffic_match_midpoint_distance_m"),
        "match_overlap": attrs.get("traffic_match_fraction_near"),
        "direction_cosine": attrs.get("traffic_match_direction_cosine"),
        "last_attempt_reason": attrs.get("traffic_last_attempt_reason"),
        "time_is_imputed": attrs.get("time_is_imputed", True),
    })

probe_results_df = pd.DataFrame(probe_rows)
display(probe_results_df)

if RUN_LIVE_TRAFFIC and batch_summary is not None:
    print("Reverse reassignments:",
          batch_summary.get("reverse_direction_reassignments"))
    print("Reverse inferred observations:",
          batch_summary.get("reverse_inferred_observed_edges"))


,edge_id,road_class,sampled,directly_queried,matched,match_source,match_reason,traffic_status,traffic_reason,current_speed_kph,free_flow_speed_kph,confidence,match_distance_m,match_overlap,direction_cosine,last_attempt_reason,time_is_imputed
0,"(26389013, 1116949690, 0)",trunk,True,True,True,direct_probe,candidate_match,yellow,observed,18.0,27.0,1.00000,0.51,1.0,0.9999,NaN,False
1,"(2180741853, 2360746811, 0)",trunk,True,True,True,direct_probe,candidate_match,yellow,observed,18.0,27.0,1.00000,5.01,1.0,0.9984,NaN,False
2,"(5898890847, 4749990829, 0)",trunk,True,True,True,direct_probe,candidate_match,clear,observed,27.0,27.0,1.00000,4.17,1.0,1.0000,NaN,False
3,"(3818359076, 3818359130, 0)",trunk,True,True,True,direct_probe,candidate_match,yellow,observed,18.0,27.0,1.00000,0.43,1.0,0.9999,NaN,False
4,"(1116949690, 2180739560, 0)",trunk,True,True,True,direct_probe,candidate_match,yellow,observed,18.0,27.0,1.00000,0.93,1.0,1.0000,NaN,False
5,"(2180741854, 2180744124, 0)",trunk,True,True,True,direct_probe,candidate_match,clear,observed,24.0,24.0,1.00000,4.61,1.0,0.9999,NaN,False
6,"(1260002921, 5940891836, 0)",trunk,True,True,True,direct_probe,candidate_match,yellow,observed,16.0,24.0,1.00000,5.45,1.0,0.9999,NaN,False
7,"(276548, 3764277424, 0)",residential,True,True,False,NaN,heading_proxy_mismatch,missing,matched_on_reverse_edge,NaN,NaN,NaN,0.51,1.0,-0.9998,matched_on_reverse_edge,True
8,"(776736207, 776736265, 0)",residential,True,True,False,NaN,midpoint_too_far,missing,midpoint_too_far,NaN,NaN,NaN,190.04,NaN,NaN,midpoint_too_far,True
9,"(154176677, 255468062, 0)",secondary,True,True,False,NaN,heading_proxy_mismatch,missing,matched_on_reverse_edge,NaN,NaN,NaN,0.50,1.0,-1.0000,matched_on_reverse_edge,True


Reverse reassignments: 2
Reverse inferred observations: 2


## 7 — Update probe history and audit the Snapshot

In [24]:

if RUN_LIVE_TRAFFIC and batch_summary is not None and batch_summary.get("requests_made", 0) > 0:
    history = update_probe_history(
        fresh_snapshot,
        batch_summary,
        history=history,
    )

    history_path = save_probe_history(history)
    print("Probe history saved:", history_path)
else:
    print("Probe history was not changed because no live collection was run.")

audit = audit_traffic_snapshot(
    G,
    snapshot,
    summary=summary,
)

print("========== TRAFFIC SNAPSHOT AUDIT ==========")
for key, value in audit.items():
    print(f"{key}: {value}")

assert audit["graph_identity_ok"], "Snapshot topology/length audit failed"
assert audit["costs_ok"], "Snapshot contains invalid routing costs"
assert audit["algorithm_smoke_test_ready"], "Snapshot is not structurally ready"

print("PASS: Snapshot passed structural audit")


Probe history saved: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/probe_history.json
========== TRAFFIC SNAPSHOT AUDIT ==========
graph_identity_ok: True
costs_ok: True
base_nodes: 2569
snapshot_nodes: 2569
total_directed_edges: 5850
sampled_directed_edges: 57
directly_queried_directed_edges: 48
matched_edges: 33
observed_edges: 33
direct_observed_edges: 24
reverse_inferred_observed_edges: 9
missing_edges: 5817
imputed_edges: 5817
invalid_status_edges: 0
observed_coverage_percent: 0.564
direct_observed_coverage_percent: 0.41
reverse_observed_coverage_percent: 0.154
algorithm_smoke_test_ready: True
final_traffic_comparison_ready: False
warnings: ['Observed traffic coverage is sparse; most travel times are not measured', 'TomTom geometry point order is not authoritative legal direction proof']
PASS: Snapshot passed structural audit


## 8 — Build routing graphs for Notebook 06

In [25]:

free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow",
)

measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured",
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured",
)

print("Free-flow graph edges:", free_flow_time.number_of_edges())
print("Measured-time graph edges:", measured_time.number_of_edges())
print("Measured-distance graph edges:", measured_distance.number_of_edges())

print("PASS: Routing graphs prepared")


Free-flow graph edges: 5850
Measured-time graph edges: 5850
Measured-distance graph edges: 5850
PASS: Routing graphs prepared


## 9 — Save the merged Snapshot Bundle



In [26]:

if RUN_LIVE_TRAFFIC and batch_summary is not None and batch_summary.get("requests_made", 0) > 0:
    previous_runs = []
    if previous_bundle is not None:
        previous_runs = list(previous_bundle.get("collection_runs", []))

    previous_runs.append({
        "finished_at_utc": summary.get("finished_at_utc"),
        "requests_made": summary.get("requests_made"),
        "request_errors": summary.get("request_errors"),
        "newly_refreshed_edges": refreshed_edges,
        "cumulative_usable_observed_edges": summary.get(
            "cumulative_usable_observed_edges"
        ),
    })

    traffic_bundle = dict(previous_bundle or {})
    traffic_bundle.update({
        "snapshot": snapshot,
        "summary": summary,
        "plan": selected_probe_records,
        "corridor_plan": plan,
        "selected_edge_ids": selected_edge_ids,
        "collection_runs": previous_runs,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    })

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path = snapshot_path.with_suffix(".tmp")

    with temporary_path.open("wb") as f:
        pickle.dump(
            traffic_bundle,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    temporary_path.replace(snapshot_path)
    save_summary(summary)

    print("PASS: Merged snapshot bundle saved:", snapshot_path)
    print("Saved usable observations:",
          count_usable_observations(snapshot)["usable_observed_edges"])
    print("Backup remains available:", backup_path if previous_snapshot is not None else "No previous snapshot")

else:
    traffic_bundle = previous_bundle
    print("No new live collection was saved in this run.")
    print("Current snapshot:", snapshot_path)


PASS: Merged snapshot bundle saved: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl
Saved usable observations: 33
Backup remains available: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot_before_merge_20261109_105318.pkl


## 10 — Optional interactive traffic map check

In [27]:

traffic_widget, ui_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
)

display(traffic_widget)


In [33]:

print("Origin node:", ui_state["origin_node"])
print("Destination node:", ui_state["destination_node"])
print("Last double-click:", ui_state["last_click_latlon"])

assert ui_state["origin_node"] in G
assert ui_state["destination_node"] in G

if ui_state["last_click_latlon"] is None:
    print("Map loaded correctly. No new double-click was made in this session.")
else:
    print("PASS: Interactive map received a double-click.")


Origin node: 26592999
Destination node: 26389375
Last double-click: (51.49634719159713, -0.16410827636718753)
PASS: Interactive map received a double-click.


In [34]:
from pprint import pprint

print("========== TOMTOM API DIAGNOSTICS ==========")
if batch_summary is None:
    print("No new live batch was run in this notebook session.")
    print("Saved snapshot summary:")
    pprint(summary)
else:
    print("\nRequests:")
    print(batch_summary.get("requests_made"))
    print("\nRequest errors:")
    print(batch_summary.get("request_errors"))
    print("\nHTTP stop status:")
    print(batch_summary.get("stopped_http_status"))
    print("\nAPI error groups:")
    pprint(batch_summary.get("api_error_groups", []))
    print("\nMatching diagnostics:")
    pprint(batch_summary.get("map_matching_diagnostics", {}))
    print("\nTraffic reasons:")
    pprint(batch_summary.get("reasons", {}))
    print("\nTraffic counts:")
    pprint(batch_summary.get("counts", {}))

========== TOMTOM API DIAGNOSTICS ==========

Requests:
12

Request errors:
0

HTTP stop status:
None

API error groups:
[]

Matching diagnostics:
{'candidate_match': 10, 'heading_proxy_mismatch': 2, 'midpoint_too_far': 2}

Traffic reasons:
{'matched_on_reverse_edge': 2,
 'midpoint_too_far': 2,
 'not_queried': 5836,
 'observed': 10}

Traffic counts:
{'clear': 3, 'closed': 0, 'missing': 5840, 'red': 0, 'yellow': 7}


In [35]:
import json
import pandas as pd
from datetime import datetime

route_probe_path = RESULTS_DIR / "route_probe_candidates.json"
print("========== ROUTE CANDIDATE FILE ==========")
if route_probe_path.exists():
    payload = json.loads(route_probe_path.read_text(encoding="utf-8"))
    print("File:", route_probe_path.resolve())
    print("Modified:", datetime.fromtimestamp(route_probe_path.stat().st_mtime).isoformat())
    print("Number of candidates:", len(payload))
    display(pd.DataFrame(payload))
else:
    print("No route candidate file exists yet. The corridor plan is still available.")

========== ROUTE CANDIDATE FILE ==========
File: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/route_probe_candidates.json
Modified: 2026-11-09T14:23:02.515599
Number of candidates: 3


,edge_id,latitude,longitude,length_m,road_class,current_traffic_status,time_is_imputed,probe_fraction
0,"[25291778, 5898995362, 0]",51.495059,-0.174919,114.34,unclassified,missing,True,0.5
1,"[6280192964, 34519894, 0]",51.496979,-0.173807,266.83,"[unclassified, living_street]",missing,True,0.5
2,"[34519894, 26389375, 0]",51.498980,-0.174206,181.39,unclassified,missing,True,0.5


In [36]:
print("========== REVERSE EDGE INSPECTION ==========")

for item in selected_probe_records:

    u, v, k = tuple(item["edge_id"])

    print("\nRequested edge:", (u, v, k))

    reverse_edges = snapshot.get_edge_data(
        v, u, default={}
    )

    for reverse_key in reverse_edges:

        reverse_id = (v, u, reverse_key)
        attrs = snapshot[v][u][reverse_key]

        print({
            "reverse_edge": reverse_id,
            "traffic_status": attrs.get("traffic_status"),
            "traffic_matched": attrs.get("traffic_matched"),
            "match_source": attrs.get("traffic_match_source"),
            "current_speed_kph": attrs.get("current_speed_kph"),
            "time_is_imputed": attrs.get("time_is_imputed"),
        })


========== REVERSE EDGE INSPECTION ==========

Requested edge: (26389013, 1116949690, 0)

Requested edge: (2180741853, 2360746811, 0)

Requested edge: (5898890847, 4749990829, 0)

Requested edge: (3818359076, 3818359130, 0)

Requested edge: (1116949690, 2180739560, 0)

Requested edge: (2180741854, 2180744124, 0)

Requested edge: (1260002921, 5940891836, 0)

Requested edge: (276548, 3764277424, 0)
{'reverse_edge': (3764277424, 276548, 0), 'traffic_status': 'clear', 'traffic_matched': True, 'match_source': 'reverse_inferred', 'current_speed_kph': 28.0, 'time_is_imputed': False}

Requested edge: (776736207, 776736265, 0)
{'reverse_edge': (776736265, 776736207, 0), 'traffic_status': 'missing', 'traffic_matched': False, 'match_source': None, 'current_speed_kph': None, 'time_is_imputed': True}

Requested edge: (154176677, 255468062, 0)
{'reverse_edge': (255468062, 154176677, 0), 'traffic_status': 'yellow', 'traffic_matched': True, 'match_source': 'reverse_inferred', 'current_speed_kph': 16.0

In [37]:
def get_usable_edge_ids(graph):

    result = set()

    if graph is None:
        return result

    for u, v, k, attrs in graph.edges(
        keys=True, data=True
    ):

        usable = (
            attrs.get("traffic_matched") is True
            and attrs.get("time_is_imputed") is False
            and attrs.get("traffic_status")
                in {"red", "yellow", "clear", "closed"}
            and attrs.get("traffic_match_source")
                in {"direct_probe", "reverse_inferred"}
        )

        if usable:
            result.add((u, v, k))

    return result


before_ids = get_usable_edge_ids(previous_snapshot)
after_ids = get_usable_edge_ids(snapshot)

new_ids = after_ids - before_ids

print("Before:", len(before_ids))
print("After:", len(after_ids))
print("New usable directed edges:", len(new_ids))

print("\nNewly observed edges:")
display(pd.DataFrame([
    {
        "edge_id": edge_id,
        "status": snapshot[edge_id[0]][edge_id[1]][edge_id[2]]
            .get("traffic_status"),
        "match_source": snapshot[edge_id[0]][edge_id[1]][edge_id[2]]
            .get("traffic_match_source"),
    }
    for edge_id in sorted(new_ids, key=str)
]))


Before: 23
After: 33
New usable directed edges: 10

Newly observed edges:


,edge_id,status,match_source
0,"(1116949690, 2180739560, 0)",yellow,direct_probe
1,"(1260002921, 5940891836, 0)",yellow,direct_probe
2,"(1667203511, 25496614, 0)",yellow,direct_probe
3,"(2180741853, 2360746811, 0)",yellow,direct_probe
4,"(2180741854, 2180744124, 0)",clear,direct_probe
5,"(255468062, 154176677, 0)",yellow,reverse_inferred
6,"(26389013, 1116949690, 0)",yellow,direct_probe
7,"(3764277424, 276548, 0)",clear,reverse_inferred
8,"(3818359076, 3818359130, 0)",yellow,direct_probe
9,"(5898890847, 4749990829, 0)",clear,direct_probe
